# 第 9 章 余剰分析と課税: 数字を変えて確かめる

**使い方**: 「Parameters」のセルの数字を書き換えて、上から順に実行します（メニューの Run → Run All Cells）。数字を変えるたびに、結果と図が変わります。

起動: アプリのディレクトリで `uv run --with jupyterlab jupyter lab notebooks/` を実行します。

記事: `docs/article/micro-basics-programming/09-surplus-and-tax.md`

In [ ]:
# --- Parameters: ここの数字を書き換えます ---
demand_intercept = 100  # 需要 Q = a + s_d·P の切片 a
demand_slope = -1       # 需要の傾き s_d（負）
supply_intercept = -10  # 供給 Q = c + s_s·P の切片 c
supply_slope = 1        # 供給の傾き s_s（正）
tax = 9                 # 従量税（売り手に、1 単位あたり課す）

In [ ]:
# --- 準備（変更しません）: lib を読み込めるようにします ---
import sys
from pathlib import Path

_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "lib").is_dir())
if str(_root) in sys.path:
    sys.path.remove(str(_root))
sys.path.insert(0, str(_root))
_loaded_lib = sys.modules.get("lib")
if _loaded_lib is not None and (
    Path(getattr(_loaded_lib, "__file__", "") or "").resolve()
    != (_root / "lib" / "__init__.py").resolve()
):
    for _name in tuple(sys.modules):
        if _name == "lib" or _name.startswith("lib."):
            del sys.modules[_name]

## 均衡と、課税のあとの結果

In [ ]:
from lib.chapter08.market import LinearMarket, equilibrium
from lib.chapter09.surplus import tax_equilibrium, tax_incidence

market = LinearMarket(demand_intercept, demand_slope, supply_intercept, supply_slope)
price, quantity = equilibrium(market)
print(f"課税前の均衡: 価格 {price:.2f}、数量 {quantity:.2f}")

result = tax_equilibrium(market, tax)
print(f"課税後: 買い手の価格 {result.buyer_price:.2f}、売り手の手取り "
      f"{result.seller_price:.2f}、数量 {result.quantity:.2f}")
print(f"税収 {result.tax_revenue:.2f}、死荷重 {result.deadweight_loss:.2f}")
buyer, seller = tax_incidence(market, tax)
print(f"税の負担: 買い手 {buyer:.2f}、売り手 {seller:.2f}")

## 図

In [ ]:
from lib.chapter09.plot import plot_tax

plot_tax(market, tax, q_max=100, p_max=100)

## 試してみよう

- `tax` を 0 にすると、税収も死荷重も 0 になります。18 にすると、死荷重は 81 です
  （9 のときの 20.25 の 4 倍。死荷重は、税額の **2 乗**に比例します）
- 需要を急にする（`demand_intercept = 72.5`、`demand_slope = -0.5`）と、買い手が多く
  負担し（6.0 対 3.0）、死荷重は 13.5 に小さくなります。需要が**非弾力的**なほど、
  税の死荷重は小さくなります
- 供給を急にする（`supply_intercept = 17.5`、`supply_slope = 0.5`）と、売り手が多く
  負担します（3.0 対 6.0）。**弾力性の小さい側**が、多く負担します